# Dashboard — Child Literacy Indicator (Gold Layer)

This notebook consumes the **3 analytical tables from the Gold layer** produced by `pipelines/batch/build_gold.py` and builds the pipeline's final consumption dashboards:

| Table | Content |
|---|---|
| `municipality_indicator` | % of literate students by municipality + year |
| `target_vs_result` | Actual result vs. target (gap), by municipality/state/national |
| `time_evolution` | Historical series of the indicator (national + by state) |

## Read strategy (with fallback)

For each table, the notebook tries, **in order**, until it manages to load the data:

1. **Amazon Athena** — `SELECT * FROM <table>` via `boto3`, using the Glue Catalog registered in `infrastructure/athena_ddl.sql` (the same serverless SQL consumption path described in the README);
2. **Direct S3** — reading the Gold Parquet files (`s3://<gold-bucket>/<table>/`) via `boto3` + `pandas`, without going through Athena;
3. **Local `--dry-run`** — Parquet files written locally to `output/gold/<table>/` by `build_gold.py --dry-run`, for development/demo without AWS.

This lets the notebook run both in an environment with the AWS infrastructure already provisioned (`infrastructure/setup_aws.sh` + `pipelines/orchestrator.py`) and locally, without credentials, using `python pipelines/orchestrator.py --dry-run`.

In [ ]:
%matplotlib inline
import sys
import time
import warnings
from io import BytesIO
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
from botocore.config import Config

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


def find_repo_root(start: Path) -> Path:
    """Walks up the directory tree from the kernel's cwd until it finds
    `pipelines/common.py` -- works whether the notebook runs from the repo
    root or from inside `notebooks/`."""
    for candidate in (start, *start.parents):
        if (candidate / "pipelines" / "common.py").exists():
            return candidate
        if (candidate / "fase2" / "pipelines" / "common.py").exists():
            return candidate / "fase2"
    raise FileNotFoundError(
        "Could not locate the repository root (folder containing pipelines/common.py) "
        f"starting from {start}"
    )


REPO_ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT / "pipelines"))

from common import (  # noqa: E402
    AWS_REGION,
    GOLD_BUCKET,
    PROJECT_NAME,
    read_parquet_from_s3,
    read_partitioned_parquet_from_s3,
)

print(f"Repository root: {REPO_ROOT}")
print(f"Project: {PROJECT_NAME} | Region: {AWS_REGION} | Gold bucket: {GOLD_BUCKET}")

## Configuration

Athena database/workgroup names follow the same defaults as `infrastructure/config.sh` (derived from `PROJECT_NAME`). All values below can be overridden via environment variable, exactly like in the pipeline scripts.

In [ ]:
import os

# Same convention as infrastructure/config.sh: the Glue database doesn't accept hyphens.
GLUE_DATABASE = os.environ.get("GLUE_DATABASE", PROJECT_NAME.replace("-", "_"))
ATHENA_WORKGROUP = os.environ.get("ATHENA_WORKGROUP", f"{PROJECT_NAME}-wg")

# Local folder with the Parquet files written by `build_gold.py --dry-run` (the script's own default).
LOCAL_GOLD_DIR = Path(os.environ.get("LOCAL_GOLD_DIR", REPO_ROOT / "output" / "gold"))

# Short timeouts: if no AWS credentials/infra are available, fail fast and fall
# back to the next source in the fallback chain instead of hanging on long retries.
FAST_FAIL_CONFIG = Config(connect_timeout=5, read_timeout=10, retries={"max_attempts": 2})

# (Gold table name, is it partitioned by year/state_code?) -- see build_gold.py / athena_ddl.sql
GOLD_TABLES = {
    "municipality_indicator": True,
    "target_vs_result": True,
    "time_evolution": False,
}

print(f"Glue database: {GLUE_DATABASE} | Athena workgroup: {ATHENA_WORKGROUP}")
print(f"Local fallback: {LOCAL_GOLD_DIR}")

## Loading the Gold tables (Athena → S3 → local `--dry-run`)

In [ ]:
# Numeric/boolean columns expected per table -- needed because reading via
# Athena comes back as CSV (everything as text), while reading via S3/local
# Parquet already preserves the original type. Applying the same coercion to
# all 3 sources guarantees the charts always receive the same dtypes.
NUMERIC_COLS = {
    "municipality_indicator": [
        "year", "literacy_rate", "avg_portuguese_score", *[f"proficiency_level_{i}_ratio" for i in range(9)],
    ],
    "target_vs_result": [
        "year", "literacy_rate",
        "target_municipality", "gap_municipality", "target_state", "gap_state", "target_national", "gap_national",
    ],
    "time_evolution": ["year", "municipalities_assessed", "avg_literacy_rate", "literacy_target", "gap"],
}
BOOL_COLS = {"target_vs_result": ["reached_target_municipality"]}
_BOOL_MAP = {"true": True, "false": False, "True": True, "False": False, True: True, False: False}


def coerce_gold_types(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    df = df.copy()
    for col in NUMERIC_COLS.get(table_name, []):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    for col in BOOL_COLS.get(table_name, []):
        if col in df.columns:
            df[col] = df[col].map(_BOOL_MAP).astype("boolean")
    if "state_code" in df.columns:
        df["state_code"] = df["state_code"].astype("string")
    if "year" in df.columns:
        df["year"] = df["year"].astype("Int64")
    return df

In [ ]:
def get_fast_client(service: str):
    return boto3.client(service, region_name=AWS_REGION, config=FAST_FAIL_CONFIG)


def run_athena_query(query: str, *, timeout_s: float = 60.0, poll_interval_s: float = 1.5) -> pd.DataFrame:
    """Runs a query on Athena via boto3 (start -> poll -> reads the result
    CSV directly from S3) and returns the result as a DataFrame."""
    athena = get_fast_client("athena")
    exec_id = athena.start_query_execution(
        QueryString=query,
        QueryExecutionContext={"Database": GLUE_DATABASE},
        WorkGroup=ATHENA_WORKGROUP,
    )["QueryExecutionId"]

    deadline = time.monotonic() + timeout_s
    execution = athena.get_query_execution(QueryExecutionId=exec_id)["QueryExecution"]
    while execution["Status"]["State"] in ("QUEUED", "RUNNING"):
        if time.monotonic() > deadline:
            athena.stop_query_execution(QueryExecutionId=exec_id)
            raise TimeoutError(f"Athena query exceeded {timeout_s:.0f}s: {query!r}")
        time.sleep(poll_interval_s)
        execution = athena.get_query_execution(QueryExecutionId=exec_id)["QueryExecution"]

    state = execution["Status"]["State"]
    if state != "SUCCEEDED":
        reason = execution["Status"].get("StateChangeReason", state)
        raise RuntimeError(f"Athena query finished with '{state}': {reason}")

    output_location = execution["ResultConfiguration"]["OutputLocation"]
    bucket, key = output_location.replace("s3://", "", 1).split("/", 1)
    body = get_fast_client("s3").get_object(Bucket=bucket, Key=key)["Body"].read()
    return pd.read_csv(BytesIO(body))


def load_from_s3(table_name: str, partitioned: bool) -> pd.DataFrame:
    s3_client = get_fast_client("s3")
    if partitioned:
        df = read_partitioned_parquet_from_s3(GOLD_BUCKET, table_name, s3_client=s3_client)
    else:
        df = read_parquet_from_s3(GOLD_BUCKET, f"{table_name}/{table_name}.parquet", s3_client=s3_client)
    if df.empty:
        raise ValueError(f"s3://{GOLD_BUCKET}/{table_name}/ is empty or does not exist")
    return df


def load_local_dry_run(table_name: str, partitioned: bool) -> pd.DataFrame:
    table_dir = LOCAL_GOLD_DIR / table_name
    if not table_dir.exists():
        raise FileNotFoundError(f"{table_dir} does not exist -- run build_gold.py --dry-run first")
    path = table_dir if partitioned else table_dir / f"{table_name}.parquet"
    return pd.read_parquet(path, engine="pyarrow")


def load_gold_table(table_name: str, partitioned: bool) -> pd.DataFrame:
    """Fallback chain: Athena -> direct S3 -> local Parquet (--dry-run).
    Returns the first source that responds successfully; prints which one was used."""
    sources = (
        ("Athena", lambda: run_athena_query(f"SELECT * FROM {table_name}")),
        ("S3 (direct Parquet)", lambda: load_from_s3(table_name, partitioned)),
        ("Local --dry-run", lambda: load_local_dry_run(table_name, partitioned)),
    )
    last_error: Exception | None = None
    for label, loader in sources:
        try:
            df = loader()
            if df is None or df.empty:
                raise ValueError("empty result")
            print(f"[OK] {table_name}: loaded via {label} ({len(df)} row(s), {len(df.columns)} column(s))")
            return coerce_gold_types(df, table_name)
        except Exception as exc:  # noqa: BLE001 -- any failure here should just trigger the next fallback
            print(f"[--] {table_name}: failed via {label} ({type(exc).__name__}: {exc}) -- trying next source")
            last_error = exc
    raise RuntimeError(f"Could not load '{table_name}' from any source (Athena/S3/local)") from last_error

In [ ]:
df_indicator = load_gold_table("municipality_indicator", GOLD_TABLES["municipality_indicator"])
df_target = load_gold_table("target_vs_result", GOLD_TABLES["target_vs_result"])
df_evolution = load_gold_table("time_evolution", GOLD_TABLES["time_evolution"])

In [ ]:
print("municipality_indicator:", df_indicator.shape)
display(df_indicator.head())

print("\ntarget_vs_result:", df_target.shape)
display(df_target.head())

print("\ntime_evolution:", df_evolution.shape)
display(df_evolution.head())

## 1. Time evolution of the indicator

Historical series of the literacy rate vs. the National Commitment to Literate Children target, based on the `time_evolution` table.

In [ ]:
national = df_evolution[df_evolution["aggregation_level"] == "national"].sort_values("year")

fig, ax = plt.subplots()
ax.plot(
    national["year"], national["avg_literacy_rate"],
    marker="o", linewidth=2.5, color="#1f77b4", label="Result (literacy rate)",
)
ax.plot(
    national["year"], national["literacy_target"],
    marker="s", linestyle="--", color="#d62728", label="Target (National Commitment)",
)
ax.fill_between(
    national["year"].astype(float), national["avg_literacy_rate"], national["literacy_target"],
    color="#d62728", alpha=0.08,
)
ax.set_title("Evolution of the Child Literacy Indicator — Brazil")
ax.set_xlabel("Year")
ax.set_ylabel("% of literate students")
ax.set_xticks(national["year"].astype(int))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
state_evolution = df_evolution[df_evolution["aggregation_level"] == "state"].sort_values(["state_code", "year"])

fig = px.line(
    state_evolution,
    x="year", y="avg_literacy_rate", color="state_code",
    markers=True,
    hover_data=["state_name", "region", "literacy_target", "gap"],
    title="Evolution of the literacy rate by state",
    labels={"avg_literacy_rate": "% literate", "year": "Year", "state_code": "State"},
)
fig.update_layout(legend_title_text="State", height=550)
fig.show()

## 2. Target vs. Result

Comparison between the actual result and the target defined for the same year, based on the `target_vs_result` table, aggregated by state for the most recent available year.

In [ ]:
available_target_years = df_target.dropna(subset=["target_state"])["year"]
latest_target_year = int(available_target_years.max()) if not available_target_years.empty else int(df_target["year"].max())

state_gap = (
    df_target[df_target["year"] == latest_target_year]
    .groupby(["state_code", "state_name"], as_index=False)
    .agg(result=("literacy_rate", "mean"), target=("target_state", "mean"), gap=("gap_state", "mean"))
    .dropna(subset=["target"])
    .sort_values("gap")
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(state_gap))
width = 0.35
ax.bar([i - width / 2 for i in x], state_gap["result"], width, label="Result", color="#1f77b4")
ax.bar([i + width / 2 for i in x], state_gap["target"], width, label="Target", color="#ff7f0e")
ax.set_xticks(list(x))
ax.set_xticklabels(state_gap["state_code"])
ax.set_ylabel("% of literate students")
ax.set_title(f"Target vs. Result by state — {latest_target_year}")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
state_gap_plot = state_gap.assign(reached_target=state_gap["gap"] >= 0).sort_values("gap")

fig = px.bar(
    state_gap_plot,
    x="state_code", y="gap", color="reached_target",
    color_discrete_map={True: "#2ca02c", False: "#d62728"},
    hover_data=["state_name", "result", "target"],
    title=f"Gap (result - target) by state — {latest_target_year}",
    labels={"gap": "Gap (p.p.)", "state_code": "State", "reached_target": "Reached the target"},
)
fig.add_hline(y=0, line_dash="dot", line_color="gray")
fig.update_layout(height=500)
fig.show()

## 3. Municipality and state ranking

Ranking based on the `municipality_indicator` table, for the most recent available year.

In [ ]:
latest_indicator_year = int(df_indicator["year"].max())
year_df = df_indicator[df_indicator["year"] == latest_indicator_year]

top15 = year_df.nlargest(15, "literacy_rate")[["municipality_name", "state_code", "literacy_rate"]]
bottom15 = year_df.nsmallest(15, "literacy_rate")[["municipality_name", "state_code", "literacy_rate"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

top15_sorted = top15.sort_values("literacy_rate")
axes[0].barh(top15_sorted["municipality_name"] + " (" + top15_sorted["state_code"] + ")", top15_sorted["literacy_rate"], color="#2ca02c")
axes[0].set_title(f"Top 15 municipalities — {latest_indicator_year}")
axes[0].set_xlabel("% of literate students")

bottom15_sorted = bottom15.sort_values("literacy_rate", ascending=False)
axes[1].barh(bottom15_sorted["municipality_name"] + " (" + bottom15_sorted["state_code"] + ")", bottom15_sorted["literacy_rate"], color="#d62728")
axes[1].set_title(f"Bottom 15 municipalities — {latest_indicator_year}")
axes[1].set_xlabel("% of literate students")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
state_ranking = (
    year_df.groupby(["state_code", "state_name", "region"], as_index=False)["literacy_rate"]
    .mean()
    .sort_values("literacy_rate", ascending=False)
)

fig = px.bar(
    state_ranking,
    x="state_code", y="literacy_rate", color="region",
    hover_data=["state_name"],
    title=f"State ranking by average literacy rate — {latest_indicator_year}",
    labels={"literacy_rate": "% literate (average)", "state_code": "State", "region": "Region"},
)
fig.update_layout(height=500, xaxis={"categoryorder": "total descending"})
fig.show()

## How to run

- **With AWS provisioned**: run `infrastructure/setup_aws.sh`, populate Gold via `python pipelines/orchestrator.py`, and run the notebook normally -- it tries Athena first and falls back to reading directly from S3 if Athena isn't available.
- **Without AWS (100% local)**: run `python pipelines/orchestrator.py --dry-run` (or just `python pipelines/batch/build_gold.py --dry-run`, if the local Silver already exists) to generate `output/gold/`, then run the notebook -- the first two sources will fail quickly (short timeout) and it automatically falls back to the local Parquet files.

To reprocess the charts with updated data, just run the pipeline again and re-run this notebook's cells -- the notebook itself doesn't cache any state to disk.